In [2]:
import cv2
import numpy as np
import mediapipe as mp
# import argparse
import os
# from datetime import datetime
from scipy.signal import find_peaks
import time
import math
import csv

In [3]:
mpDraw = mp.solutions.drawing_utils
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence = 0.8)

# Drawing style helpers (optional customizations)
DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(255,0,0), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)

In [4]:
# requirements = [
#     "opencv-python",
#     "mediapipe",
#     "numpy"
# ]

# with open("requirements.txt", "w") as f:
#     for r in requirements:
#         f.write(r + "\n")

# print("requirements.txt created!")

In [5]:
def score_jump_distance(distance_cm):
    if distance_cm >= 160:
        return 5
    elif 140 <= distance_cm <= 159:
        return 4
    elif 120 <= distance_cm <= 139:
        return 3
    elif 100 <= distance_cm <= 119:
        return 2
    else:
        return 1


In [6]:
def calc_angle(a, b, c):
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])

    ba = a - b
    bc = c - b

    cosang = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    ang = np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0)))
    return ang


In [7]:
def arm_takeoff(lms):
    ls = lms[11]
    le = lms[13]
    lw = lms[15]
    rs = lms[12]
    re = lms[14]
    rw = lms[16]

    left_angle = calc_angle(ls, le, lw)
    right_angle = calc_angle(rs, re, rw)
    avg_angle = (left_angle + right_angle) / 2
    return avg_angle


In [8]:
def score_arm_takeoff(avg_angle):
    if 70 <= avg_angle <= 110:
        return 5
    elif 60 <= avg_angle <= 120:
        return 4
    elif 50 <= avg_angle <= 130:
        return 3
    elif 40 <= avg_angle <= 140:
        return 2
    else:
        return 1

In [9]:
def flight_phase(lms):
    
    lh = lms[23]
    lk = lms[25]
    la = lms[27]

    rh = lms[24]
    rk = lms[26]
    ra = lms[28]

    left_knee_lift = lh.y - lk.y
    right_knee_lift = rh.y - rk.y
    knee_lift = max(left_knee_lift, right_knee_lift)

    return knee_lift


In [10]:
def score_flight_phase(knee_lift):
    if knee_lift > 0.10:
        return 5
    elif knee_lift > 0.08:
        return 4
    elif knee_lift > 0.06:
        return 3
    elif knee_lift > 0.04:
        return 2
    else:
        return 1
    

In [11]:
def landing_stability(lms):
    la = lms[27]
    ra = lms[28]

    ankle_diff = abs(la.y - ra.y)
    
    return ankle_diff


In [12]:
def score_landing_stability(ankle_diff):
    
    if ankle_diff < 0.02:
        return 5
    elif ankle_diff < 0.04:
        return 4
    elif ankle_diff < 0.06:
        return 3
    elif ankle_diff < 0.10:
        return 2
    else:
        return 1

In [13]:
def composite_technique_score(a, f, l):
    total = a + f + l

    if total >= 13:
        return total, 5
    elif total >= 10:
        return total, 4
    elif total >= 7:
        return total, 3
    elif total >= 4:
        return total, 2
    else:
        return total, 1


In [14]:
def predict_category(_score, sprint_score):
    
    # Safety checks
    if _score < 0: _score = 0
    if _score > 20: _score = 20
    if sprint_score < 0: sprint_score = 0
    if sprint_score > 5: sprint_score = 5

    # -------------------------------
    # Top Tier Category (Excellent)
    # -------------------------------
    if _score >= 17:
        if sprint_score >= 4:
            return "Excellent"
        else:
            return "Above Average"

    # -------------------------------
    # High-Mid Tier (Above Average / Average)
    # -------------------------------
    elif 14 <= _score <= 17:
        if sprint_score >= 3:
            return "Above Average"
        elif sprint_score == 2:
            return "Average"
        else:
            return "Below Average"

    # -------------------------------
    # Mid Tier (Average / Below Average)
    # -------------------------------
    elif 10 <= _score <= 13:
        if sprint_score >= 3:
            return "Average"
        elif sprint_score == 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lower Tier (Below Average)
    # -------------------------------
    elif 6 <= _score <= 9:
        if sprint_score >= 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lowest Tier (Poor)
    # -------------------------------
    else:  # _score 0–5
        return "Poor"

In [15]:
def add_data(row):
    # row must be a list: ["value1", "value2", ...]
    with open("long_jump.csv", "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(row)

In [16]:
open("long_jump.csv", "w").close()
data = ["ID", "Name", "Video_path", "Jumped_distance_cm", "Avg_angle_final", "Knee_lift_final", "Ankle_diff_final", "Score_18", "Category"]
add_data(data)

In [20]:
def long_jump(ID="DCXXXX", name="Life", path=0, data_print='Y', jumped_length = 60):
    
    # cap = cv2.VideoCapture(0)  # 0 = default camera
    # path = "img-5610-0q6jn2tf_fZX3eQZg.mov"
    cap = cv2.VideoCapture(path)


    # Define video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # remove in live case
    fps = cap.get(cv2.CAP_PROP_FPS)                            # frames per second
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))       # total frames
    # video_time = frame_count / fps

    # print("FPS:", fps)
    # print("Total Frames:", frame_count)
    # print("Video Duration (seconds):", video_time_seconds)


    # save the output video
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # out = cv2.VideoWriter(video_filename, fourcc, fps, (frame_width, frame_height))

    left_line_x = 0.05
    right_line_x = 0.87
    
    # all pose and pose-score append here     # will store (posture, arms, legs)
    pose_all = []
    pose_scores = [] 

    # Sprint time + score
    sprint_time = []
    sprint_score = 0

    # change acc to your threshold
    start_frame_idx = 0      # example 
    finish_frame_idx = fps   # example

    # total score
    _score = 0

    frame_idx = 0
    start_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # Convert BGR → RGB for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Run pose detection
        results = pose.process(rgb)

        if results.pose_landmarks:
            # Enumerate all landmarks
            # for id, lm in enumerate(results.pose_landmarks.landmark):
            #     # Optional: Draw skeleton
            #     mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
                
            #     # Convert normalized landmark to pixel coordinates
            #     h, w, c = frame.shape
            #     cx, cy = int(lm.x * w), int(lm.y * h)
            #     # IDs to highlight: 23, 24, 25, 26
            #     if id in [23, 24, 25, 26]:
            #         # Draw circle on frame
            #         cv2.circle(frame, (cx, cy), 8, (0, 255, 0), -1)
            
        
            mpDraw.draw_landmarks(frame, results.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
            lms = results.pose_landmarks.landmark

            h, w, c = frame.shape
            cx1 = int(left_line_x * w)
            cx2 = int(right_line_x * w)
            cy1 = int(0.48 * h)
            cy2 = int(0.48 * h)
            cv2.circle(frame, (cx1, cy1), 8, (0, 255, 0), -1)
            cv2.circle(frame, (cx2, cy2), 8, (0, 255, 0), -1)
            cx = int((cx1+cx2)/2)
            cy = int((cy1+cy2)/2)
            cv2.circle(frame, (cx, cy), 8, (0, 0, 255), -1)

            # arm takeoff
            avg_angle = arm_takeoff(lms)
            a = score_arm_takeoff(avg_angle)

            # flight phase
            knee_lift = flight_phase(lms)
            f = score_flight_phase(knee_lift)

            # landing stability
            ankle_diff = landing_stability(lms)
            l = score_landing_stability(ankle_diff)

            # append all values
            pose_all.append((avg_angle, knee_lift, ankle_diff))
            pose_scores.append((a, f, l))

            # Optional: draw text on frame
            cv2.putText(frame, f"avg_angle: {avg_angle:.2f}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"knee_lift: {knee_lift*frame_height:.2f}", (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"ankle_diff: {ankle_diff*frame_height:.2f}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
            

        # out.write(frame)
        # cv2.imshow("Press 'q' to stop early", frame)
        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    # out.release()
    cv2.destroyAllWindows()

    # Aggregate all pose
    if pose_all:
        avg_angle_final = round(np.mean([a for a, _, _ in pose_all]), 2)
        knee_lift_final = round(np.mean([b for _, b, _ in pose_all]), 2)
        ankle_diff_final = round(np.mean([c for _, _, c in pose_all]), 2)


    # Aggregate technique scores
    if pose_scores:
        avg_angle_score = np.mean([a for a, _, _ in pose_scores])
        knee_lift_score = np.mean([f for _, f, _ in pose_scores])
        ankle_diff_score = np.mean([l for _, _, l in pose_scores])

        # Total score / 18
        _score = round((avg_angle_score + knee_lift_score + ankle_diff_score), 2) 

    # Sprint time + score
    # sprint_time = compute_sprint_time_from_frames(start_frame_idx, finish_frame_idx, fps)
    distance_cm = jumped_length
    jump_dist_score = score_jump_distance(distance_cm)

    category = predict_category(_score, jump_dist_score)

    # show data
    if (data_print =='Y' or data_print == 'y'):
        print(f"Jumped distance: {distance_cm:.2f}")
        print(f"Technique – arms:--> avg_angle_final: {avg_angle_final}")
        print(f"Technique – legs:--> knee_lift_final: {knee_lift_final*frame_height:.2f}")
        print(f"Technique – ankle:--> ankle_diff_final: {ankle_diff_final*frame_height:.2f}")
        print(f"Technique total points: {_score}/18 and Category:--> {category}")
        

    # adding data
    data = [ID, name, path, distance_cm, avg_angle_final, knee_lift_final, ankle_diff_final, _score, category]
    add_data(data)

    # print(_score, "--", category)
    return _score, category

In [21]:
# ID = input("(DCXXXXX)Enter unique ID:")
# name = input("Name of Candidate:")
# path = input("Path of your Video:")
# # "demo-shuttle-run_HwmEdnDL.mp4"
# data_print = input("Wants to print data Y/N:")
# shuttle_score, category = long_jump(ID, name, path, data_print)
# print("printing result here-------------->>")
# print(f"Your High Knee Jump Score is--> {shuttle_score:.2f}/18 and Category--> {category}")

In [22]:
ID = "DC0001"
name = "Nayan"
# path = "20m_running/20m Shuttle Run Test_720p (online-video-cutter.com).mp4"
# path = "Test_Run.mp4"
# path = "Test_Run.mp4"
path = "20251223_154622.mp4"
data_print = "Y"
# data_print = input("Wants to print data Y/N:")
jumped_length = 60
meter_score, category = long_jump(ID, name, path, data_print, jumped_length)

Jumped distance: 60.00
Technique – arms:--> avg_angle_final: 152.73
Technique – legs:--> knee_lift_final: -151.20
Technique – ankle:--> ankle_diff_final: 21.60
Technique total points: 6.72/18 and Category:--> Poor
